# Bucketing

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

In [ ]:
spark_path = "E:\\Spark\\spark-3.2.0-bin-hadoop2.7"

import findspark
findspark.init(spark_path)

In [ ]:
from pyspark.storagelevel import StorageLevel
from pyspark.sql.types import *
import pyspark.sql.functions as F
from pyspark.sql import SparkSession

In [ ]:
spark = (
    SparkSession
    .builder
    .config("spark.driver.memory", "10g")
    .master("local[*]")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("ERROR")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [ ]:
orders_file = "../data/bucketing/orders.csv"
df_orders = spark.read.csv(orders_file, header=True, inferSchema=True)

In [ ]:
df_orders.show(5, False)
df_orders.printSchema()

In [ ]:
products_file = "../data/bucketing/products.csv"
df_products = spark.read.csv(products_file, header=True, inferSchema=True)

In [ ]:
df_products.show(5, False)
df_products.printSchema()

In [ ]:
df_products.select("product_id").distinct().count()

In [ ]:
df_orders.select("order_id").distinct().count()

## Bucketing In Joins

In [ ]:
df_orders_product_details = (
    df_orders.join(
        df_products,
        on="product_id",
        how="inner"
    )
)

In [ ]:
df_orders_product_details.explain()

In [ ]:
df_orders_product_details.count()

In [ ]:
(
    df_products
    .write.bucketBy(4, col="product_id")
    .mode("overwrite")
    .saveAsTable("products_bucketed")
)

In [ ]:
(
    df_orders
    .write.bucketBy(4, col="product_id")
    .mode("overwrite")
    .saveAsTable("orders_bucketed")
)

In [ ]:
df_orders_bucketed = spark.table("orders_bucketed")
df_products_bucketed = spark.table("products_bucketed")

In [ ]:
df_orders_product_details_bucketed = (
    df_orders_bucketed.join(
        df_products_bucketed,
        on="product_id",
        how="inner"
    )
)

In [ ]:
df_orders_product_details_bucketed.explain()

In [ ]:
df_orders_product_details_bucketed.count()

## Bucketing In Aggregations

In [ ]:
df_orders.show(5, False)

In [ ]:
# WITHOUT BUCKETING

df_product_sales = (
    df_orders
    .groupBy("product_id")
    .agg(F.sum("total_amount").alias("sales"))
)

df_product_sales.explain()

In [ ]:
# WITH BUCKETING

df_product_sales = (
    df_orders_bucketed
    .groupBy("product_id")
    .agg(F.sum("total_amount").alias("sales"))
)

df_product_sales.explain()

## Bucket Pruning

In [ ]:
df_product_sales_bucket_pruning = (
    df_orders_bucketed
    .filter(F.col("product_id") == 1)
    .groupBy("product_id")
    .agg(F.sum("total_amount").alias("sales"))
)

In [ ]:
df_product_sales_bucket_pruning.explain()